# GW170817 mass-ratio prior $P(q)$

Builds the 1D marginal posterior $P(d_{GW}\mid q)$, $q = m_2/m_1 \le 1$, needed for the black-hole-collapse hypothesis likelihood (Eq. 23 of the BH-hypothesis note). Uses the same raw LIGO/Virgo low-spin `PhenomPNRT` posterior samples that `lv.py` already uses to build `LV_prior.h5` for $\Lambda_1,\Lambda_2$ — just a different pair of columns, so no new external data is needed.

Output: `LV_prior_q.h5` with datasets `x` (q grid) and `data` ($P(q)$), matching the naming convention of `LV_prior.h5` so the C++ loader can reuse the same `read_hdf5_vector`-based reader.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

In [ ]:
DATA_FILE = "low_spin_PhenomPNRT_posterior_samples.dat"
OUTPUT_FILE = "LV_prior_q.h5"
N_GRID = 200

# Columns (see lv.py):
# 0 costheta_jn  1 luminosity_distance_Mpc  2 m1_detector_frame_Msun
# 3 m2_detector_frame_Msun  4 lambda1  5 lambda2  6 spin1  7 spin2  8 costilt1  9 costilt2
data = np.loadtxt(DATA_FILE, skiprows=1)
m1 = data[:, 2]
m2 = data[:, 3]

# q is dimensionless, so the detector-frame vs. source-frame distinction
# (both scaled by the same (1+z)) cancels.
q = m2 / m1

print(f"{len(q)} samples")
print(f"q range: [{q.min():.4f}, {q.max():.4f}]")
assert np.all(q <= 1.0), "expected m2 <= m1 convention (q <= 1)"

In [ ]:
# Kernel density estimate of P(q). q has bounded support on (0, 1], so check
# the diagnostic plot below for boundary bias near q=1 (KDE will smear
# density past the boundary). If that looks bad, reflect the samples about
# q=1 before fitting (q_reflected = 2 - q, concatenate, then keep grid <= 1
# and double the resulting density) rather than switching bandwidth blindly.
kde = gaussian_kde(q)

q_grid = np.linspace(0.0, 1.0, N_GRID)
p_q = kde(q_grid)
p_q /= np.trapz(p_q, q_grid)

print(f"KDE bandwidth factor: {kde.factor:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.hist(q, bins=50, density=True, alpha=0.4, label="posterior samples")
ax.plot(q_grid, p_q, color="C1", label="KDE")
ax.set_xlabel(r"$q = m_2/m_1$")
ax.set_ylabel(r"$P(q)$")
ax.set_xlim(0.0, 1.0)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
with h5py.File(OUTPUT_FILE, "w") as f:
    f.create_dataset("x", data=q_grid)
    f.create_dataset("data", data=p_q)

print(f"wrote {OUTPUT_FILE}")